# Misinformation Detection Pipeline — With Evidence Augmentation

Changes from previous version:
- `build_dataset_with_evidence`: wires up Google Search → score_documents → augment_knowledge_graph per article
- Disk cache for search results: re-running cells never re-fetches the same query
- Overfitting fixes: dropout increased, weight decay, early stopping, gradient clipping
- n_topics reduced to 3 per article (saves API quota)

In [1]:
import nltk
try:
    nltk.download('punkt_tab', quiet=True)
except:
    nltk.download('punkt', quiet=True)

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

ENCODER = SentenceTransformer('all-MiniLM-L6-v2')
EMBEDDING_DIM = 384

## Step 1: Fact Extraction & Knowledge Graph

In [2]:
import networkx as nx
from nltk.tokenize import sent_tokenize

def extract_facts(text: str) -> list[tuple[str, np.ndarray]]:
    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    if not sentences:
        return []
    embeddings = ENCODER.encode(sentences, batch_size=32, show_progress_bar=False)
    return list(zip(sentences, embeddings))


def build_knowledge_graph(facts: list[tuple[str, np.ndarray]], threshold: float = 0.75) -> nx.Graph:
    G = nx.Graph()
    if not facts:
        return G
    sentences, embeddings = zip(*facts)
    embeddings = np.array(embeddings)
    for i, (sent, emb) in enumerate(zip(sentences, embeddings)):
        G.add_node(i, text=sent, embedding=emb, source='input', weight=1.0)
    sim_matrix = cosine_similarity(embeddings)
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sim_matrix[i, j] > threshold:
                G.add_edge(i, j, similarity=float(sim_matrix[i, j]))
    return G

## Step 2: Topic Extraction + Google Search with Disk Cache

The cache saves every search result to a local JSON file keyed by query string.
Re-running this cell or restarting the kernel will load from disk instead of
calling the API again — important because Google's free tier is 100 queries/day.

In [3]:
import requests
import json
import os
import hashlib
from sklearn.feature_extraction.text import TfidfVectorizer

CACHE_DIR = 'search_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_path(query: str) -> str:
    # Use a hash of the query as the filename so special characters don't cause issues
    key = hashlib.md5(query.encode()).hexdigest()
    return os.path.join(CACHE_DIR, f'{key}.json')


def google_search(query: str, api_key: str, cse_id: str, num_results: int = 10) -> list[dict]:
    """
    Search Google, with transparent disk caching.
    First call for a query hits the API and saves results.
    Every subsequent call for the same query loads from disk.
    """
    cache_file = _cache_path(query)
    if os.path.exists(cache_file):
        with open(cache_file, 'r') as f:
            return json.load(f)

    url = 'https://www.googleapis.com/customsearch/v1'
    params = {'q': query, 'key': api_key, 'cx': cse_id, 'num': num_results}
    response = requests.get(url, params=params)
    response.raise_for_status()
    results = response.json().get('items', [])

    with open(cache_file, 'w') as f:
        json.dump(results, f)
    return results


def extract_topics(text: str, n_topics: int = 3) -> list[str]:
    """
    TF-IDF keyword extraction. n_topics=3 (down from 5) to reduce API calls.
    200 articles x 3 topics = 600 queries total.
    """
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=100)
    sentences = sent_tokenize(text)
    if len(sentences) < 2:
        sentences = [text]
    tfidf_matrix = vectorizer.fit_transform(sentences)
    feature_names = vectorizer.get_feature_names_out()
    scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
    top_indices = scores.argsort()[::-1][:n_topics]
    return [feature_names[i] for i in top_indices]


def collect_evidence(text: str, api_key: str, cse_id: str, k: int = 10) -> list[dict]:
    topics = extract_topics(text, n_topics=3)
    seen_links = set()
    all_results = []
    for topic in topics:
        try:
            results = google_search(topic, api_key, cse_id, num_results=k)
        except requests.HTTPError as e:
            print(f"  Search failed for '{topic}': {e}")
            continue
        for r in results:
            if r['link'] not in seen_links:
                seen_links.add(r['link'])
                all_results.append(r)
    return all_results

## Step 3: Cross-Source Agreement Scoring & Ranking

In [4]:
def score_documents(documents: list[dict]) -> list[tuple[dict, float]]:
    """
    Rank documents by cross-source agreement.
    Each document's score = average cosine similarity to all other documents.
    Documents whose content is corroborated by many others score highest.
    """
    snippets = [doc.get('snippet', '') for doc in documents]
    if not snippets:
        return []
    embeddings = ENCODER.encode(snippets, batch_size=32, show_progress_bar=False)
    sim_matrix = cosine_similarity(embeddings)
    n = len(documents)
    scores = []
    for i, doc in enumerate(documents):
        avg_sim = (sim_matrix[i].sum() - 1.0) / max(n - 1, 1)
        scores.append((doc, float(avg_sim)))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


def blend_reliability(scored_docs, reliability_scores: dict, alpha: float = 0.7):
    blended = []
    for doc, agreement_score in scored_docs:
        reliability = reliability_scores.get(doc.get('link', ''), 0.5)
        combined = alpha * agreement_score + (1 - alpha) * reliability
        blended.append((doc, combined))
    blended.sort(key=lambda x: x[1], reverse=True)
    return blended

## Step 4: Knowledge Graph Augmentation with Evidence

In [5]:
def augment_knowledge_graph(G: nx.Graph,
                             scored_docs: list[tuple[dict, float]],
                             sim_threshold: float = 0.75,
                             score_threshold: float = 0.3) -> nx.Graph:
    """
    Add evidence nodes from ranked documents into the graph.
    New evidence nodes are weighted by their document's agreement score,
    so highly-corroborated sources pull harder on the graph representation.
    Collects all changes first, then applies them to avoid mutating G during iteration.
    """
    existing_nodes = list(G.nodes(data=True))  # snapshot
    next_id = max(G.nodes()) + 1 if G.nodes() else 0

    new_nodes = []
    new_edges = []

    for doc, doc_score in scored_docs:
        if doc_score < score_threshold:
            continue
        snippet = doc.get('snippet', '')
        if not snippet:
            continue
        evidence_facts = extract_facts(snippet)
        for fact_text, fact_emb in evidence_facts:
            nid = next_id
            next_id += 1
            new_nodes.append((nid, fact_text, fact_emb, doc_score))
            fact_emb_2d = fact_emb.reshape(1, -1)
            for existing_id, data in existing_nodes:
                existing_emb = data['embedding'].reshape(1, -1)
                sim = float(cosine_similarity(fact_emb_2d, existing_emb)[0, 0])
                if sim > sim_threshold:
                    new_edges.append((nid, existing_id, sim, doc_score))

    for nid, text, emb, score in new_nodes:
        G.add_node(nid, text=text, embedding=emb, source='evidence', weight=score)
    for nid, existing_id, sim, doc_score in new_edges:
        G.add_edge(nid, existing_id, similarity=sim, evidence_weight=doc_score)

    return G

## Step 5: GCN Classifier

Overfitting fixes applied vs previous version:
- Dropout raised from 0.3 → 0.5
- Weight decay (L2 regularization) kept in Adam
- Early stopping: training halts if validation loss stops improving
- Gradient clipping: prevents exploding gradients on small batches
- BCEWithLogitsLoss replaces BCE + Sigmoid (numerically more stable)

In [6]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data


def graph_to_pyg(G: nx.Graph, label: int | None = None) -> Data:
    node_ids = list(G.nodes())
    id_map = {nid: i for i, nid in enumerate(node_ids)}
    x = torch.tensor(
        np.array([G.nodes[n]['embedding'] * G.nodes[n].get('weight', 1.0) for n in node_ids]),
        dtype=torch.float
    )
    if G.edges():
        edges = [(id_map[u], id_map[v]) for u, v in G.edges()]
        edge_index = torch.tensor(edges, dtype=torch.long).T
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    data = Data(x=x, edge_index=edge_index)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data


class FakeNewsGCN(torch.nn.Module):
    """
    Two-layer GCN with global mean pooling.
    Now uses raw logits + BCEWithLogitsLoss (more numerically stable than
    sigmoid inside the model + BCELoss, which can hit floating point issues
    near 0 and 1).
    """
    def __init__(self, input_dim: int = EMBEDDING_DIM, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, 64)
        self.classifier = torch.nn.Linear(64, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)  # raw logits — no sigmoid here


def train_gcn(train_data: list[Data],
              val_data: list[Data],
              epochs: int = 100,
              lr: float = 1e-3,
              patience: int = 15) -> FakeNewsGCN:
    """
    Train with early stopping on validation loss.
    patience=15 means training stops if val loss doesn't improve for 15 epochs.
    This is the primary fix for the overfitting seen in the previous run.
    """
    model = FakeNewsGCN()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = torch.nn.BCEWithLogitsLoss()  # sigmoid + BCE fused, numerically stable
    train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=16)

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_state = None

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch).squeeze()
            loss = criterion(out, batch.y.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()
            train_loss += loss.item()

        # --- Validation ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                out = model(batch.x, batch.edge_index, batch.batch).squeeze()
                val_loss += criterion(out, batch.y.squeeze()).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}  train_loss={avg_train:.4f}  val_loss={avg_val:.4f}")

        # --- Early stopping ---
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            epochs_no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1} (no val improvement for {patience} epochs)")
                break

    # Restore the best weights, not the final (potentially overfit) ones
    model.load_state_dict(best_state)
    return model


def evaluate_gcn(model: FakeNewsGCN, pyg_dataset: list[Data]) -> dict:
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    model.eval()
    loader = DataLoader(pyg_dataset, batch_size=16)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch).squeeze()
            probs = torch.sigmoid(logits)  # convert logits to probabilities at eval time
            all_preds.extend(probs.tolist())
            all_labels.extend(batch.y.squeeze().tolist())
    binary_preds = [1 if p > 0.5 else 0 for p in all_preds]
    return {
        'accuracy': accuracy_score(all_labels, binary_preds),
        'f1': f1_score(all_labels, binary_preds),
        'auc': roc_auc_score(all_labels, all_preds)
    }

## Step 6: Model Persistence

In [7]:
def save_model(model: torch.nn.Module, path: str):
    torch.save(model.state_dict(), path)
    print(f"Saved to {path}")

def load_model(path: str) -> FakeNewsGCN:
    model = FakeNewsGCN()
    model.load_state_dict(torch.load(path, weights_only=True))
    model.eval()
    return model

## Step 7: Dataset Building — With Evidence Augmentation

Set your API credentials here. The cache means each unique topic query is only
fetched once — safe to re-run cells without burning quota.

In [8]:
# Add your credentials here
GOOGLE_API_KEY = 'YOUR_API_KEY_HERE'
GOOGLE_CSE_ID  = 'YOUR_CSE_ID_HERE'

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

df = pd.read_csv(os.path.abspath('data/fake_news_dataset.csv'))
df['label_binary'] = (df['label'] == 'fake').astype(int)
print(f"Dataset: {len(df)} articles | Class balance: {df['label_binary'].value_counts().to_dict()}")


def build_dataset_with_evidence(df: pd.DataFrame,
                                 api_key: str,
                                 cse_id: str,
                                 text_col: str = 'text',
                                 label_col: str = 'label_binary',
                                 sample: int | None = None) -> list[Data]:
    """
    Full pipeline per article:
      1. Extract facts + build base knowledge graph from article text
      2. Collect Google Search evidence for the article's key topics
      3. Score documents by cross-source agreement
      4. Augment the graph with weighted evidence nodes
      5. Convert to PyG Data object

    Falls back to the unamented graph if search fails or returns nothing,
    so the dataset build won't crash mid-way through.
    """
    if sample:
        df = df.sample(sample, random_state=42)

    pyg_data = []
    n_augmented = 0
    n_fallback = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building graphs'):
        text = row[text_col]
        label = int(row[label_col])

        # Step 1: Base graph from article
        facts = extract_facts(text)
        if not facts:
            continue
        G = build_knowledge_graph(facts)

        # Steps 2-4: Evidence augmentation
        try:
            documents = collect_evidence(text, api_key, cse_id, k=10)
            if documents:
                scored_docs = score_documents(documents)
                G = augment_knowledge_graph(G, scored_docs)
                n_augmented += 1
            else:
                n_fallback += 1  # search returned nothing, use base graph
        except Exception as e:
            print(f"  Evidence collection failed: {e}")
            n_fallback += 1

        # Step 5: Convert to PyG format
        pyg_data.append(graph_to_pyg(G, label=label))

    print(f"\nBuilt {len(pyg_data)} graphs | Augmented: {n_augmented} | Fallback (no evidence): {n_fallback}")
    return pyg_data


pyg_dataset = build_dataset_with_evidence(
    df,
    api_key=GOOGLE_API_KEY,
    cse_id=GOOGLE_CSE_ID,
    sample=200
)

Dataset: 20000 articles | Class balance: {1: 10056, 0: 9944}


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Search failed for 'final': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=final&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10
  Search failed for 'today': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=today&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10


Building graphs:   0%|          | 1/200 [00:00<02:07,  1.56it/s]

  Search failed for 'south': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=south&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10


...

  Search failed for 'realize': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=realize&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10
  Search failed for 'father': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=father&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10
  Search failed for 'watch': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=watch&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10
  Search failed for 'return': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=return&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10
  Search failed for 'fly': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=fly&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10


Building graphs: 100%|█████████▉| 199/200 [01:28<00:00,  2.72it/s]

  Search failed for 'statement': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=statement&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10
  Search failed for 'method': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=method&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10


Building graphs: 100%|██████████| 200/200 [01:29<00:00,  2.25it/s]

  Search failed for 'mother': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=mother&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10
  Search failed for 'expect': 400 Client Error: Bad Request for url: https://www.googleapis.com/customsearch/v1?q=expect&key=YOUR_API_KEY_HERE&cx=YOUR_CSE_ID_HERE&num=10

Built 200 graphs | Augmented: 0 | Fallback (no evidence): 200


## Step 8: Train / Validate / Test

We now do a 3-way split: train / validation / test.
Validation is used for early stopping. Test is held out until the very end.

In [11]:
# 70% train, 15% val, 15% test
train_data, temp_data = train_test_split(pyg_dataset, test_size=0.3, random_state=42)
val_data, test_data   = train_test_split(temp_data,   test_size=0.5, random_state=42)
print(f"Split: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test")

model = train_gcn(train_data, val_data, epochs=100, patience=15)

metrics = evaluate_gcn(model, test_data)
print(f"\nTest Results:")
print(f"  Accuracy: {metrics['accuracy']:.3f}")
print(f"  F1 Score: {metrics['f1']:.3f}")
print(f"  AUC-ROC:  {metrics['auc']:.3f}")

save_model(model, 'fake_news_gcn.pt')

Split: 140 train | 30 val | 30 test
Epoch  10  train_loss=0.6580  val_loss=0.7405
Early stopping at epoch 16 (no val improvement for 15 epochs)

Test Results:
  Accuracy: 0.533
  F1 Score: 0.000
  AUC-ROC:  0.496
Saved to fake_news_gcn.pt


## Next Steps

1. **NLI-typed edges**: Use `cross-encoder/nli-deberta-v3-small` to label edges as
   'entailment', 'contradiction', or 'neutral'. The GCN could then learn different
   aggregation behavior per edge type — contradicted claims are a strong fake signal.

2. **DBSCAN consensus clustering**: Replace average similarity ranking with clustering.
   The largest cluster = consensus. Documents outside it = outliers.

3. **Graph Attention Networks (GAT)**: `GATConv` learns which edges to weight more
   heavily. Often outperforms plain GCN for heterogeneous graphs like ours (mix of
   input and evidence nodes with different reliability weights).

4. **Scale up**: Once the pipeline is validated on 200 articles, run on the full dataset.
   The search cache means this doesn't re-cost API quota for already-seen queries.